# Lineare Regression: Trefferquote → Wahlbeteiligung

Die Mathematik liegt in `georep.analysis.regression` (`scipy.stats.linregress`); dieses Notebook lädt das
Ergebnis der Pipeline und zeichnet. Vorher einmal `uv run georep bootstrap` und `uv run georep run`.

Der ursprüngliche Code der Arbeit folgte dem Data-Science-Skript von Dr. Völkel
(https://voelkel.pages.cs.uni-duesseldorf.de/ds-skript/lineare-regression.html) und rechnete die Gerade
von Hand, mit `curve_fit`, `linregress` und `sklearn.LinearRegression` nach; die Übereinstimmung der vier
Wege ist seit Phase 2 ein Test (`tests/golden/test_regression_numbers.py`, `test_polars_parity.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from georep.analysis.regression import regress
from georep.layout import Layout
from georep.settings import load_settings

layout = Layout.from_settings(load_settings())
results = pl.read_parquet(layout.results)
x = results["trefferquote"].to_numpy()
y = (results["wahlbeteiligung"] * 100).to_numpy()

full = regress(results)
trimmed = regress(results, exclude=(1, 10))
print("alle Wahlbezirke:", full.summary())
print("ohne 1 und 10:   ", trimmed.summary())

In [ ]:
def f(x, beta_0, beta_1):
    return beta_1 * x + beta_0


y_hat = f(x, full.intercept, full.slope)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(x, y, "+", label="Daten")
ax.plot(x, y_hat, label="Gerade minimalen Abstands")
yerr = np.stack((np.maximum(0, y - y_hat), -np.minimum(0, y - y_hat)))
ax.errorbar(x, y, yerr=yerr, linewidth=0, elinewidth=1, label="Residuen")
ax.set_xlabel("Trefferquote (Treffer je Wahlberechtigtem)")
ax.set_ylabel("Wahlbeteiligung in Prozent")
ax.legend()
plt.show()

In [ ]:
residuals = y - y_hat
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(y_hat, residuals)
ax.axhline(0, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Vorhergesagte Wahlbeteiligung (ŷ)")
ax.set_ylabel("Residuen (y - ŷ)")
plt.tight_layout()
plt.show()